<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/evaluation/geomind_retrieval_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluating GeoMind Retrieval with LlamaIndex

This example uses GeoMind as a structured web corpus for retrieval evaluation. It loads a small sample of knowledge pages with `SimpleWebPageReader`, builds a `VectorStoreIndex`, and uses GeoMind's explicit cross-reference links as page-level relevance judgments.

The example intentionally uses 20 pages to keep runtime small. Ground-truth relationships are restricted to links whose targets are also present in this sample, so the reported metrics describe this sample rather than the complete GeoMind knowledge graph.


If you're opening this notebook on Colab, install the required LlamaIndex packages first.


In [ ]:
%pip install llama-index llama-index-readers-web llama-index-embeddings-huggingface

## Load a GeoMind sample

Collect the first 20 knowledge-page URLs from GeoMind's public knowledge index, then load them as LlamaIndex documents. Each document keeps a normalized `page_id` based on its URL path so relationships can be compared consistently.


In [ ]:
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup

from llama_index.core import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.readers.web import SimpleWebPageReader

KNOWLEDGE_INDEX = "https://shanhai-geo.github.io/knowledge/"
SAMPLE_SIZE = 20

response = requests.get(KNOWLEDGE_INDEX, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

urls = []
for link in soup.find_all("a", href=True):
    absolute_url = urljoin(KNOWLEDGE_INDEX, link["href"])
    path = urlparse(absolute_url).path

    if (
        path.startswith("/knowledge/")
        and path.endswith(".html")
        and absolute_url not in urls
    ):
        urls.append(absolute_url)

urls = urls[:SAMPLE_SIZE]
print(f"Selected {len(urls)} pages")

reader = SimpleWebPageReader(html_to_text=True, fail_on_error=True)
documents = reader.load_data(urls=urls)

for document in documents:
    document.metadata["page_id"] = urlparse(document.metadata["url"]).path

print(f"Loaded {len(documents)} documents")

/home/rudra/open/llama_index/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Selected 20 pages
Loaded 20 documents


## Build page-level ground truth

GeoMind pages contain explicit cross-reference links. For each sampled page, collect links to other pages that are also in the sample. These links act as the relevance judgments for retrieval evaluation.


In [ ]:
selected_page_ids = {document.metadata["page_id"] for document in documents}

relationships = {}

for document in documents:
    source_url = document.metadata["url"]
    source_id = document.metadata["page_id"]

    response = requests.get(source_url, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    linked_page_ids = set()
    for link in soup.find_all("a", href=True):
        absolute_url = urljoin(source_url, link["href"])
        path = urlparse(absolute_url).path

        if path in selected_page_ids and path != source_id:
            linked_page_ids.add(path)

    relationships[source_id] = linked_page_ids

pages_with_ground_truth = sum(
    bool(related) for related in relationships.values()
)
print(
    f"Pages with ground-truth links: {pages_with_ground_truth}/{len(documents)}"
)

Pages with ground-truth links: 20/20


## Build the vector index

Use a local multilingual embedding model because the GeoMind pages contain Chinese text. No LLM or API key is required for this retrieval-only evaluation.


In [ ]:
embed_model = HuggingFaceEmbedding(model_name="intfloat/multilingual-e5-small")

index = VectorStoreIndex.from_documents(
    documents,
    embed_model=embed_model,
)

retriever = index.as_retriever(similarity_top_k=10)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 566.78it/s]


## Evaluate retrieval against cross-references

A webpage may be split into multiple nodes, so retrieved results are deduplicated by `page_id`. The source page itself is excluded. `Hit@K` checks whether at least one linked page appears in the top K unique pages, while `Recall@K` measures the fraction of linked pages recovered in the top K.


In [ ]:
def evaluate_retrieval(
    source_id,
    query,
    retriever,
    relationships,
    k_values=(1, 3, 5),
):
    results = retriever.retrieve(query)
    retrieved_ids = []

    for result in results:
        page_id = result.node.metadata["page_id"]

        if page_id == source_id:
            continue

        if page_id not in retrieved_ids:
            retrieved_ids.append(page_id)

    relevant_ids = relationships[source_id]
    metrics = {}

    for k in k_values:
        top_k = retrieved_ids[:k]
        hits = set(top_k) & relevant_ids

        metrics[f"hit@{k}"] = int(bool(hits))
        metrics[f"recall@{k}"] = (
            len(hits) / len(relevant_ids) if relevant_ids else 0.0
        )

    return retrieved_ids, metrics

In [ ]:
evaluation_results = []

for source_id, relevant_ids in relationships.items():
    if not relevant_ids:
        continue

    document = next(
        doc for doc in documents if doc.metadata["page_id"] == source_id
    )

    # Use the beginning of each source page as a lightweight semantic query.
    query = document.text[:300]

    _, metrics = evaluate_retrieval(
        source_id=source_id,
        query=query,
        retriever=retriever,
        relationships=relationships,
    )

    evaluation_results.append(
        {
            "source_id": source_id,
            **metrics,
        }
    )

print(f"Evaluated {len(evaluation_results)} pages")

Evaluated 20 pages


## Aggregate results

Average the page-level metrics across all sampled pages that have at least one in-sample cross-reference.


In [ ]:
metric_names = [
    "hit@1",
    "hit@3",
    "hit@5",
    "recall@1",
    "recall@3",
    "recall@5",
]

average_metrics = {
    metric: sum(result[metric] for result in evaluation_results)
    / len(evaluation_results)
    for metric in metric_names
}

for metric, value in average_metrics.items():
    print(f"{metric}: {value:.3f}")

hit@1: 0.250
hit@3: 0.600
hit@5: 0.700
recall@1: 0.083
recall@3: 0.200
recall@5: 0.250


The metrics provide a compact check of how well semantic retrieval recovers GeoMind's explicit cross-reference structure. Because this example uses a small sample and treats links as relevance judgments, the values should be interpreted as an illustrative retrieval check rather than a comprehensive benchmark of GeoMind or LlamaIndex.
